In [2]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline
from google.colab import drive  # google.colab.drive를 drive.으로 사
drive.mount('/content/drive') # 경로고정/ 이후에는 /content/drive/MyDrive/ drive 내에 경로지정

import os # 운영체제 관련기능이라는데 여기서는 경로 합칠때사용
import scipy.io as sio  # 주로 mat  파일 읽는 라이브러리  (wav dat bin)
import matplotlib.pyplot as plt # 플롯팅 라이브러리
import numpy as np # 행렬용라이브러리
# load할 데이터가많아서 따로 함수로 처리 함
# 구상은 이렇게 train 을 15개 전부올리자 ㅇ
angle_table=np.array([[90,-20+40,-90+30,-90+60,-70+60,-70+20,-70+20,-70+70,-70+20,-70+20,-60+70,-60+20,-80+20,-70+70,-70+20,-70+20],
                     [0,-20+70,-90+30,-90+30,-70+70,-70+20,-70,-70+40,-70+40,-70+10,-60+60,-60+20,-80,-70+80,-70+5,-70+10],
                     [90,-20+90,-90+10,-90+10,-70+90,-70+80,-70+30,-70+90,-70+90,-70+20,-60+90,-60+70,-80+30,-70+90,-70+70,-70+30]],dtype=np.float32) # 뒤에 dtype?
# array는 .shape  list는 len() : [] [] 행또는 열 만 확인가능..
def load_mat_dataset(folder_path,start_idx,end_idx):
  X_list=[] # 빈 list
  Y_list=[]
  for i in range(start_idx,end_idx+1):
     path = os.path.join(folder_path, f"sample{i:02d}.mat") # sample{i} 대산 sample{i:02d}로해서 01 02 03 이렇게표현
     mat=sio.loadmat(path,struct_as_record=False, squeeze_me=True)
     epo=mat["epo"] # epo는 내 matlab에서 이름이 epo임 ..
     x=epo.x
     y=epo.y
     x=np.transpose(x,(2,1,0))
     x=x.astype(np.float32)
     #z-score 정규화 axis 2가 시간축
     mean=np.mean(x,axis=2,keepdims=True)
     std=np.std(x,axis=2,keepdims=True) + 1e-6
     x=(x-mean)/std
     # 라벨정리
     class_label=np.argmax(y,axis=0) # 1
     angle_label=angle_table[class_label]
     X_list.append(x) # x값을 X_list에 계속누적 누적 sample1: 1~150 trial먼저누적  다음 151~300 누적
     Y_list.append(angle_label) # Y_list : 총 trial(2250)  * 16 (angle각)
     print(f"{path}")
     print("x:",x.shape," angle_label:",angle_label.shape)
  # concatenate: list 안에있는 numpy array 변수를 하나로합칠떄..
  X=np.concatenate(X_list,axis=0) # list 를 np용 array로만들면서 한번에 합침
                                  # np.cocatenate a로 바로하면 [X x] 이런식으로 계속 누적해줘야해서 비효율적이래  마지막에한번
  Y=np.concatenate(Y_list,axis=0)
  return X,Y

train_X,train_Y=load_mat_dataset("/content/drive/MyDrive/Training set",1,5)
#훈련먼저
import torch
from torch import nn # torch.nn=nn.
from torch.utils.data import TensorDataset,DataLoader,random_split # torch.utils.data =데이터학습용 라이브러리
device = "cuda" if torch.cuda.is_available() else "cpu" # 런타임 -> 런타임유형에서 gpu로 바꿔야한데 속도빠르게용 print(torch.cuda.is_available()) /print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
#numpy array -> Tensor 행렬 (from_numpy 행렬공유 .tensor 새로운행렬)
X_tensor=torch.from_numpy(train_X).float() #.double or float or half or int or long or byte...
Y_tensor=torch.from_numpy(train_Y).float()


batch_size=32
#TensorDataset: 같은번호끼리 묶어서 dataset이라는 새로운 객체로 만듬
#Data_Loader: 데이터를 batch_size 만큼꺼내서 훈련시키는역할
train_loader=DataLoader(TensorDataset(X_tensor,Y_tensor),batch_size=batch_size,shuffle=True) #

print("train 개수:", len(train_dataset))






NameError: name 'torch' is not defined

**Learn the Basics** \|\| [Quickstart](quickstart_tutorial.html) \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| [Build
Model](buildmodel_tutorial.html) \|\|
[Autograd](autogradqs_tutorial.html) \|\|
[Optimization](optimization_tutorial.html) \|\| [Save & Load
Model](saveloadrun_tutorial.html)

Learn the Basics
================

Authors: [Suraj Subramanian](https://github.com/subramen), [Seth
Juarez](https://github.com/sethjuarez/), [Cassie
Breviu](https://github.com/cassiebreviu/), [Dmitry
Soshnikov](https://soshnikov.com/), [Ari
Bornstein](https://github.com/aribornstein/)

Most machine learning workflows involve working with data, creating
models, optimizing model parameters, and saving the trained models. This
tutorial introduces you to a complete ML workflow implemented in
PyTorch, with links to learn more about each of these concepts.

We\'ll use the FashionMNIST dataset to train a neural network that
predicts if an input image belongs to one of the following classes:
T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker,
Bag, or Ankle boot.

[This tutorial assumes a basic familiarity with Python and Deep Learning
concepts.]{.title-ref}

Running the Tutorial Code
-------------------------

You can run this tutorial in a couple of ways:

-   **In the cloud**: This is the easiest way to get started! Each
    section has a \"Run in Google Colab\" link at the top, which opens
    an integrated notebook in Google Colab with the code in a
    fully-hosted environment.
-   **Locally**: This option requires you to set up PyTorch and
    TorchVision first on your local machine ([installation
    instructions](https://pytorch.org/get-started/locally/)). Download
    the notebook or copy the code into your favorite IDE.

How to Use this Guide
---------------------

If you\'re familiar with other deep learning frameworks, check out the
[0. Quickstart](quickstart_tutorial.html) first to quickly familiarize
yourself with PyTorch\'s API.

If you\'re new to deep learning frameworks, head right into the first
section of our step-by-step guide: [1. Tensors](tensorqs_tutorial.html).

::: {.toctree maxdepth="2" hidden=""}
quickstart\_tutorial tensorqs\_tutorial data\_tutorial
transforms\_tutorial buildmodel\_tutorial autogradqs\_tutorial
optimization\_tutorial saveloadrun\_tutorial
:::
